In [1]:
import os
os.chdir("..")
from t2Interp.T2I import T2IModel
from t2Interp.concept_search import KSteer
from t2Interp.mapper import MLPMapper
from utils.utils import preprocess_image
import torch as th
from reporting.config_loader import load_config, wandb_init_kwargs
from utils.runningstats import WandbUpdater
from utils.training import TrainingSpec, Training
from utils.output_manager import OutputManager
from PIL import Image

/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/nirmal/miniconda3/envs/viz/lib/python3.11/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `PYTORCH_TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
model = T2IModel("CompVis/stable-diffusion-v1-4", device="cuda:0", dtype="float16")

2025-10-22 09:07:07.288 | INFO     | t2Interp.T2I:__init__:108 - Enforcing eager attention implementation for attention pattern tracing. The HF default would be to use sdpa if available. To use sdpa, set attn_implementation='sdpa' or None to use the HF default.
Keyword arguments {'attn_implementation': 'eager', 'tokenizer_kwargs': {}, 'trust_remote_code': False} are not expected by StableDiffusionPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 16.28it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases t

In [3]:
preprocess_fn = lambda x: preprocess_image(x, 512)
race_labels={"East Asian":0,"Indian":1,"Black":2,"White":3,"Middle Eastern":4,"Latino_Hispanic":5,"Southeast Asian":6}
gt_processing_fn = lambda x: th.tensor(race_labels[x], dtype=th.long)

In [4]:
model._wrappers

{'text_encoder_2': blocks:
 0: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 1: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 2: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 3: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 4: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 5: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 6: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 7: in_ | attn_in | attn_out | WO_in | WO_out | mlp_in | mlp_out | out_ | q_in | k_in | v_in | q_out | k_out | v_out
 8: in_ | attn_in | attn_out | WO_in 

### check latent shape

In [5]:
cache = model.run_with_cache([""], accessors= [model.unet_2.down_attn_blocks[0].self_attn_out], **{"num_inference_steps":1, "seed":40})
for k,v in cache.items():
    print(k, v.output.shape)

100%|██████████| 1/1 [00:00<00:00,  3.00it/s]

unet_down_block_attn0_self_attn_out torch.Size([2, 4096, 320])


In [ ]:
kwargs = {
    "preprocess_fn": preprocess_fn,
    "gt_processing_fn" : gt_processing_fn,
    # "subset" :100,
    "val_split":"val",
    "dataset_column": "image",
    "ground_truth_column":"race",
    "use_val": True,
    "steps":200,
    "denoising_step":0,
    "training_device" : "cuda:0",
    "data_device" : "cpu",
    "autocast_dtype" : th.bfloat16,
    "d_submodule": 4096*320,
    "log_steps": 5,
    "refresh_batch_size":64,  
    "out_batch_size":16,
    "use_memmap": True,
    "cache_activations": True,
    "run_name":"training_race_steering_mlp"
}

dataset = "nirmalendu01/fairface-trainval-race-balanced-200"
accessor = model.unet_2.down_attn_blocks[0].self_attn_out
mapper = MLPMapper(input_dim=4096*320, hidden_dim=4096, output_dim=7)
loss_fn = th.nn.CrossEntropyLoss()
optimizers = [th.optim.Adam(mapper.parameters(), lr=1e-5)]
steering = KSteer(model)

wandb_config = load_config("reporting/config.yaml")
wandb_config["wandb"].update({"run_name" : "training_race_steering_mlp"})
init_kwargs = wandb_init_kwargs(wandb_config)
stats_updaters = [WandbUpdater(init_kwargs=init_kwargs)]

output_manager = OutputManager(**kwargs)
callbacks = [output_manager.write_metadata,output_manager.save_best_ckpt]

model.pipeline.set_progress_bar_config(disable=True)
spec = TrainingSpec(name="training_race_steering_mlp", fn = steering.fit, stats_updaters=stats_updaters,
                    callback_fns=callbacks, 
                    args=(dataset, accessor, mapper,loss_fn ,optimizers), kwargs=kwargs)
Training(spec).run_trainer()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


AssertionError: OutputManager requires a run_name argument

### ksteer - using the learnt classifier

In [ ]:
from utils.text_image_buffer import TextImageActivationBuffer
from dictionary_learning.utils import hf_dataset_to_generator
from t2Interp.concept_search import KSteer
from utils.text_img_util import run_with_hook, OutputAlterHook, replace_policy
from utils.utils import BatchIterator
from itertools import tee

def get_activations_steer_save(model:T2IModel,dataset,accessor,mapper,**kwargs):
    gen_images, gen_buffer = tee(BatchIterator(hf_dataset_to_generator(dataset,**kwargs),batch_size=kwargs.get("out_batch_size",1)))
    d_sub = kwargs.pop("d_submodule", kwargs.pop("d_submodule", None))
    buffer = TextImageActivationBuffer(gen_buffer, model, accessor,d_submodule=d_sub,refresh_batch_size=1,out_batch_size=1, **kwargs) 
    ksteer= KSteer(model)
    generate_kwargs = {}
    if "guidance_scale" in kwargs:
        generate_kwargs["guidance_scale"] = kwargs["guidance_scale"]
    for images, activations in zip(gen_images, iter(buffer)):
        print(len(images), activations.shape)
        steered_activation = ksteer.steer(activations,target_idx=[1], mapper = mapper,)
        hook_obj = OutputAlterHook(replace_policy(steered_activation))
        output = run_with_hook(model, images, accessor.module, hook_obj, accessor.io_type)
        imgs = output.images
        break
    return imgs

In [7]:
kwargs = {
    "preprocess_fn": preprocess_fn,
    "dataset_column": "image",
    "steps":1,
    "denoising_step":0,
    "data_device" : "cpu",
    "autocast_dtype" : th.bfloat16,
    "d_submodule": 4096*320,
    "refresh_batch_size":64,  
    "out_batch_size":16,
    "run_name":"training_race_steering_mlp",
    "num_inference_steps":1,
}

model.pipeline.set_progress_bar_config(disable=True)

dataset = 'nirmalendu01/fairface-trainval-race-balanced-200'
accessor = model.unet_2.down_attn_blocks[0].self_attn_out
mapper = MLPMapper(input_dim=4096*320, hidden_dim=4096, output_dim=7).to("cuda:0")
mapper.load_state_dict(th.load("./runs/steer_mlp_train_steering_20251021-050122/artifacts/best_ckpt.pt"))

img = get_activations_steer_save(model,dataset,accessor,mapper, **kwargs)

16 torch.Size([16, 1310720])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])
torch.Size([16, 1310720]) torch.Size([16, 4096, 320])